# WP1.5 - Volt/Var control

This notebook demonstrates the operation of the multiconductor power flow with volt/var control for smart inverters. The proof is based on the tests defined in the WP1.5 specification document. All results have been compared against openDSS to confirm their correctness.

Here the needed libraries and functions used for the tests are imported.

In [1]:
import multiconductor as mc
import multiconductor.test.testing_toolbox
from multiconductor.control import add_volt_var_control, check_ctr_sgen_stable_state, QVCurve
from pandapower.control import run_control
import pandas as pd
pd.set_option('display.float_format', '{:.12f}'.format) 
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)  
warnings.filterwarnings("ignore", category=DeprecationWarning)
import copy

In [9]:
grids = mc.test.testing_toolbox.load_all_test_grids("wp1_5_volt_var_control")

In [10]:
l = []
for name, net in grids.items():
    net = copy.deepcopy(net)
    mc.run_pf(net, run_control=True)
    c = multiconductor.test.testing_toolbox.compare_to_dss(net)
    l.append(c)
    
pd.DataFrame(l, index=grids.keys()).sort_index(key=lambda x: x.str.len())

,max_vm_pu_diff,max_va_degree_diff,max_i_from_ka_diff,max_i_to_ka_diff,max_p_from_mw_diff,max_p_to_mw_diff,max_q_from_mvar_diff,max_q_to_mvar_diff
twenty_bus_system_voltage_in_tolerance_limits_voltvar_control.pkl,0.000000009211,0.000000775841,0.000000011716,0.000000011716,0.000000002387,0.000000002458,0.000000001250,0.000000001152
twenty_bus_system_voltage_above_tolerance_limits_voltvar_control.pkl,0.000000031308,0.000005013474,0.000000088534,0.000000088534,0.000000017427,0.000000018105,0.000000013135,0.000000012050
twenty_bus_system_voltage_below_tolerance_limits_voltvar_control.pkl,0.000000058468,0.000004283510,0.000000042699,0.000000042697,0.000000010217,0.000000009914,0.000000020112,0.000000020540
twenty_bus_system_voltage_far_above_tolerance_limits_voltvar_control.pkl,0.000000035405,0.000005220698,0.000000088424,0.000000088424,0.000000017716,0.000000018766,0.000000013808,0.000000012131
twenty_bus_system_voltage_far_below_tolerance_limits_voltvar_control.pkl,0.000000014419,0.000007017862,0.000000119188,0.000000119188,0.000000027180,0.000000025430,0.000000012129,0.000000012961
four_bus_system_symmetric_voltage_in_tolerance_limits_voltvar_control.pkl,0.000000000000,0.000000019367,0.000000070471,0.000000070471,0.000000000369,0.000000000373,0.000000000050,0.000000000035
four_bus_system_asymmetric_voltage_in_tolerance_limits_voltvar_control.pkl,0.000000007773,0.000000046913,0.000000037362,0.000000037362,0.000000318237,0.000000321492,0.000001392039,0.000001403432
four_bus_system_symmetric_voltage_above_tolerance_limits_voltvar_control.pkl,0.000000000029,0.000000039989,0.000000358407,0.000000358407,0.000000073427,0.000000078117,0.000000043618,0.000000027204
four_bus_system_symmetric_voltage_below_tolerance_limits_voltvar_control.pkl,0.000000000030,0.000000022623,0.000000216354,0.000000216354,0.000000011146,0.000000010718,0.000000004395,0.000000002900
four_bus_system_asymmetric_voltage_above_tolerance_limits_voltvar_control.pkl,0.000000122788,0.000002035927,0.000000697918,0.000000697918,0.000005798199,0.000005679938,0.000027582476,0.000027168562


# Controller white-box test

Prior to the specified testing, the volt-var controller was tested using a 2-bus system with only an external grid connected to the first bus and a controller-equipped 3-phased generator (asymmetric_sgen) to the second bus. Both buses were connected with a line with r:x ratio of 1:10. The sgen was configured to deliver symmetric active power of 120 MW at a rated nominal power of also 120 MW. The sgen did not infeed any reactive power prior to control application. The controller's control curve was chosen according to the specification document figure 2 as is the case for all the tests in this notebook. With this setup, the voltage (vm_pu) of the external grid was modified within a continuous range of (0.87, 1.1) and stepwidth 0.001, and power flows without and with controller were executed for each of the steps. The sgen's active and reactive power output response to the voltage was recorded. 

The following graph shows the resulting voltage at the sgen bus vs. the external grid voltage for both uncontrolled case (red) and controlled case (green). Voltage at the sgen bus is always a bit higher than at the external grid because of the sgen's active power infeed. The dotted and dashed lines represent the lower and upper threshold voltages of the piecewise linear control curve. It can be seen that the controller (green curve) moved the voltage closer towards the acceptable range as soon as it left the deadband (0.967 p.u. < V < 1.033 p.u).

<img src="volt_var_control_sweeptest_voltages.png" width="800">


The following graph shows the sgen's active power output (upper curve) and reactive power output (lower curve) in percent of the nominal rated apparent power, depending on the voltage at the sgen bus. It can be seen that the reactive power output exactly follows the control curve, whereas active power is derated in order to keep the total sgen apparent power output within the allowed limit set by the rated apparent power. 

<img src="volt_var_control_sweeptest_powers.png" width="800">

# Controller validation function 

In order to validate the controller operation, the function "check_ctr_sgen_stable_state" was designed to check if the sgen power output represents a valid operation point that lies on the control curve, and respects the nominal power limit:

In [12]:
net = copy.deepcopy(grids["four_bus_system_symmetric_voltage_below_tolerance_limits_voltvar_control.pkl"])

print(f"grid contains {len(net.controller)} controller(s):")
for o in net.controller.object:
    print("    ", o)

print()
print("--- Run without controllers ---")
mc.run_pf(net, run_control=False)
check_ctr_sgen_stable_state(net, sgen_first_lvl_idx=0, verbose=True)
print("asymmetric_sgen:")
print(net.asymmetric_sgen[["p_mw", "q_mvar"]])
print(net.res_bus[["vm_pu"]].loc[3])

print()
print("--- Run with controllers and re-check ---")
mc.run_pf(net, run_control=True)
check_ctr_sgen_stable_state(net, sgen_first_lvl_idx=0, verbose=True)
print()
print("asymmetric_sgen:")
print(net.asymmetric_sgen[["p_mw", "q_mvar"]])
print()
print(net.res_bus[["vm_pu"]].loc[3])

grid contains 1 controller(s):
     Volt/Var Controller(element_index=[(0, 0), (0, 1), (0, 2)], q_model=QModelQVCurve, saturate_sn_mva=[20.], damping_coef=4)

--- Run without controllers ---
check_ctr_sgen_stable_state: sgen reactive power (0.0) does not equal target (1.0265200424839804)!
asymmetric_sgen:
                        p_mw         q_mvar
index circuit                              
0     0       1.000000000000 0.000000000000
      1       1.000000000000 0.000000000000
      2       1.000000000000 0.000000000000
               vm_pu
phase               
0     0.000000000000
1     0.958958926334
2     0.958958926334
3     0.958958926334

--- Run with controllers and re-check ---

asymmetric_sgen:
                        p_mw         q_mvar
index circuit                              
0     0       1.000000000000 0.616456344205
      1       1.000000000000 0.616456344205
      2       1.000000000000 0.616456344205

               vm_pu
phase               
0     0.000000000000
1 

As can be seen the Q values of the sgen were altered by the cotroller.

In order to create a volt/var controller one can use the function add_volt_var_control:

In [14]:
net = copy.deepcopy(grids["four_bus_system_symmetric_voltage_below_tolerance_limits_voltvar_control.pkl"])
# delete existing controllers
net.controller.drop(net.controller.index, inplace=True)

qv_curve = QVCurve(vm_points_pu=[0.92, 0.967, 1.033, 1.07], q_points_pu=[0.6, 0, 0, -0.6])
add_volt_var_control(net, element=0, qv_curve=qv_curve, et='asymmetric_sgen', mode='gang', damping_coef=4)

print(f"grid contains {len(net.controller)} controller(s):")
for o in net.controller.object:
    print("    ", o)

mc.run_pf(net, run_control=True)
check_ctr_sgen_stable_state(net, sgen_first_lvl_idx=0, verbose=True)
net.asymmetric_sgen

grid contains 1 controller(s):
     Volt/Var Controller(element_index=[(0, 0), (0, 1), (0, 2)], q_model=QModelQVCurve, saturate_sn_mva=[20.], damping_coef=4)


name  bus  from_phase  to_phase           p_mw         q_mvar  \
index circuit                                                                  
0     0        None    3           1         0 1.000000000000 0.881776493069   
      1        None    3           2         0 1.000000000000 0.881776493069   
      2        None    3           3         0 1.000000000000 0.881776493069   

               vm_pu  const_z_percent_p  const_i_percent_p  const_z_percent_q  \
index circuit                                                                   
0     0          NaN     0.000000000000     0.000000000000     0.000000000000   
      1          NaN     0.000000000000     0.000000000000     0.000000000000   
      2          NaN     0.000000000000     0.000000000000     0.000000000000   

               const_i_percent_q          sn_mva        scaling  in_service  \
index circuit                                                                 
0     0           0.000000000000 20.000000000000 1.000000000000        True   
      1           0.000000000000 20.000000000000 1.000000000000        True   
      2           0.000000000000 20.000000000000 1.000000000000        True   

               type current_source  slack control_mode  \
index circuit                                            
0     0        None            NaN  False     Volt/Var   
      1        None            NaN  False     Volt/Var   
      2        None            NaN  False     Volt/Var   

                                                   control_curve  
index circuit                                                     
0     0        <multiconductor.control.volt_var_control.QVCur...  
      1        <multiconductor.control.volt_var_control.QVCur...  
      2        <multiconductor.control.volt_var_control.QVCur...

## Tests on 4-bus meshed toplogy, 3-phase system, load or sgen at bus 2 and controlled sgen at bus 3
These tests serve to demonstrate the correct operation of volt/var control of a single sgen in the following scenarios:
1. Moderate sgen power injections and bus3 voltage is well within the specified tolerance limits
2. High sgen power injections, bus3 voltage is above the threshold and controller should bring it down
3. High or load consumption at bus2, bus3 voltage is below the threshold and controller should increase it
4. Asymmetric sgen power injection, verification of control in asymmetric conditions.

With the following command we re-load the original grids (one of them was modified by previous code)

In [15]:
grids = mc.test.testing_toolbox.load_all_test_grids("wp1_5_volt_var_control")

## Test 1: 4 bus grid with one controlled sgen, symmetric load and sgen, voltage within tolerance limits
No control action. No reactive power infeed from controlled sgen, no active power change. 

In [17]:
net = grids["four_bus_system_symmetric_voltage_in_tolerance_limits_voltvar_control.pkl"]
mc.run_pf(net)
check_ctr_sgen_stable_state(net, sgen_first_lvl_idx=1, verbose=True)

print("--- Pre/Post control comparison ---")
multiconductor.test.testing_toolbox.print_sgen_stats(net, "", [1], header=False, before_control=True) 
print()
mc.run_pf(net, run_control=True)
multiconductor.test.testing_toolbox.print_sgen_stats(net, "", [1], header=False, before_control=False)
print("\n--- Post control results ---")
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
Before control
Sgen bus: 3
vm_pu: [1.0114, 1.0114, 1.0114]
sn_mva_limit: [5.000000, 5.000000, 5.000000]
sn_mva: [5.000000, 5.000000, 5.000000]
p_mw: [5.000000, 5.000000, 5.000000]
q_mvar: [0.000000, 0.000000, 0.000000]

After control
Sgen bus: 3
vm_pu: [1.0114, 1.0114, 1.0114]
sn_mva_limit: [5.000000, 5.000000, 5.000000]
sn_mva: [5.000000, 5.000000, 5.000000]
p_mw: [5.000000, 5.000000, 5.000000]
q_mvar: [0.000000, 0.000000, 0.000000]

--- Post control results ---
Max overall vm_pu difference: 3.0242475190789264e-13
Max overall va_degree difference: 1.9366764949259985e-08
Max overall i_from_ka difference: 7.047121530767697e-08
Max overall i_to_ka difference: 7.047121530767697e-08
Max overall p_from_mw difference: 3.6851766083145776e-10
Max overall p_to_mw difference: 3.729070385816158e-10
Max overall q_from_mvar difference: 5.010991621645644e-11
Max overall q_to_mvar difference: 3.475233989469473e-11

bus results
                  vm_pu_mc      vm_pu_

## Test 2: 4 bus grid with one controlled sgen, symmetric load and sgen, voltage below tolerance limits
Controller to increase voltage towards tolerance band by reactive power infeed, possible active power deration

In [18]:
net = grids["four_bus_system_symmetric_voltage_below_tolerance_limits_voltvar_control.pkl"]
mc.run_pf(net)
check_ctr_sgen_stable_state(net, sgen_first_lvl_idx=0, verbose=True)

print("--- Pre/Post control comparison ---")
multiconductor.test.testing_toolbox.print_sgen_stats(net, "", [0], header=False, before_control=True) 
print()
mc.run_pf(net, run_control=True)
multiconductor.test.testing_toolbox.print_sgen_stats(net, "", [0], header=False, before_control=False)
print("\n--- Post control results ---")
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

check_ctr_sgen_stable_state: sgen reactive power (0.0) does not equal target (1.0265200424839804)!
--- Pre/Post control comparison ---
Before control
Sgen bus: 3
vm_pu: [0.9590, 0.9590, 0.9590]
sn_mva_limit: [20.000000, 20.000000, 20.000000]
sn_mva: [1.000000, 1.000000, 1.000000]
p_mw: [1.000000, 1.000000, 1.000000]
q_mvar: [0.000000, 0.000000, 0.000000]

After control
Sgen bus: 3
vm_pu: [0.9622, 0.9622, 0.9622]
sn_mva_limit: [20.000000, 20.000000, 20.000000]
sn_mva: [1.174742, 1.174742, 1.174742]
p_mw: [1.000000, 1.000000, 1.000000]
q_mvar: [0.616456, 0.616456, 0.616456]

--- Post control results ---
Max overall vm_pu difference: 2.976385804487336e-11
Max overall va_degree difference: 2.2623082074346712e-08
Max overall i_from_ka difference: 2.1635359814808908e-07
Max overall i_to_ka difference: 2.1635359814808908e-07
Max overall p_from_mw difference: 1.1145619538410756e-08
Max overall p_to_mw difference: 1.0718377296825565e-08
Max overall q_from_mvar difference: 4.395435126980374e-09


## Test 3: 4 bus grid with one controlled sgen, symmetric load and sgen, voltage above tolerance limits
Controller to decrease voltage towards tolerance band by reactive power infeed, possible active power deration 

In [19]:
net = grids["four_bus_system_symmetric_voltage_above_tolerance_limits_voltvar_control.pkl"]
mc.run_pf(net)
check_ctr_sgen_stable_state(net, sgen_first_lvl_idx=1, verbose=True)

print("--- Pre/Post control comparison ---")
multiconductor.test.testing_toolbox.print_sgen_stats(net, "", [1], header=False, before_control=True) 
print()
mc.run_pf(net, run_control=True)
multiconductor.test.testing_toolbox.print_sgen_stats(net, "", [1], header=False, before_control=False)
print("\n--- Post control results ---")
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

check_ctr_sgen_stable_state: sgen reactive power (0.0) does not equal target (-0.9106225117790157)!
--- Pre/Post control comparison ---
Before control
Sgen bus: 3
vm_pu: [1.0375, 1.0375, 1.0375]
sn_mva_limit: [25.000000, 25.000000, 25.000000]
sn_mva: [25.000000, 25.000000, 25.000000]
p_mw: [25.000000, 25.000000, 25.000000]
q_mvar: [0.000000, 0.000000, 0.000000]

After control
Sgen bus: 3
vm_pu: [1.0353, 1.0353, 1.0353]
sn_mva_limit: [25.000000, 25.000000, 25.000000]
sn_mva: [24.998449, 24.998449, 24.998449]
p_mw: [24.994286, 24.994286, 24.994286]
q_mvar: [-0.456210, -0.456210, -0.456210]

--- Post control results ---
Max overall vm_pu difference: 2.8543167829298e-11
Max overall va_degree difference: 3.9989085820479886e-08
Max overall i_from_ka difference: 3.584065364492517e-07
Max overall i_to_ka difference: 3.584065364492517e-07
Max overall p_from_mw difference: 7.342745789173932e-08
Max overall p_to_mw difference: 7.81171536345937e-08
Max overall q_from_mvar difference: 4.36178204665

## Test 4: 4 bus grid with one controlled sgen, symmetric load and sgen, voltage far above tolerance limits
Controller to decrease voltage towards tolerance band by high reactive power infeed, possible active power deration 

In [20]:
net = grids["four_bus_system_symmetric_voltage_far_above_tolerance_limits_voltvar_control.pkl"]
mc.run_pf(net)
check_ctr_sgen_stable_state(net, sgen_first_lvl_idx=1, verbose=True)

print("--- Pre/Post control comparison ---")
multiconductor.test.testing_toolbox.print_sgen_stats(net, "", [1], header=False, before_control=True) 
print()
mc.run_pf(net, run_control=True)
multiconductor.test.testing_toolbox.print_sgen_stats(net, "", [1], header=False, before_control=False)
print("\n--- Post control results ---")
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

check_ctr_sgen_stable_state: sgen reactive power (0.0) does not equal target (-5.852574003465191)!
--- Pre/Post control comparison ---
Before control
Sgen bus: 3
vm_pu: [1.0571, 1.0571, 1.0571]
sn_mva_limit: [30.000000, 30.000000, 30.000000]
sn_mva: [30.000000, 30.000000, 30.000000]
p_mw: [30.000000, 30.000000, 30.000000]
q_mvar: [0.000000, 0.000000, 0.000000]

After control
Sgen bus: 3
vm_pu: [1.0440, 1.0440, 1.0440]
sn_mva_limit: [30.000000, 30.000000, 30.000000]
sn_mva: [29.938962, 29.938962, 29.938962]
p_mw: [29.819501, 29.819501, 29.819501]
q_mvar: [-2.671857, -2.671857, -2.671857]

--- Post control results ---
Max overall vm_pu difference: 1.449822484289598e-10
Max overall va_degree difference: 2.9389616074126934e-08
Max overall i_from_ka difference: 4.1511685822115396e-07
Max overall i_to_ka difference: 4.1511685822115396e-07
Max overall p_from_mw difference: 4.9787651335009286e-08
Max overall p_to_mw difference: 5.3005358324753615e-08
Max overall q_from_mvar difference: 1.21593

## Test 5: 4 bus grid with one controlled sgen, asymmetric load and sgen, voltage within tolerance limits
No control action. No reactive power infeed from controlled sgen, no active power change. 

In [21]:
net = grids["four_bus_system_asymmetric_voltage_in_tolerance_limits_voltvar_control.pkl"]
mc.run_pf(net)
check_ctr_sgen_stable_state(net, sgen_first_lvl_idx=1, verbose=True)

print("--- Pre/Post control comparison ---")
multiconductor.test.testing_toolbox.print_sgen_stats(net, "", [1], header=False, before_control=True) 
print()
mc.run_pf(net, run_control=True)
multiconductor.test.testing_toolbox.print_sgen_stats(net, "", [1], header=False, before_control=False)
print("\n--- Post control results ---")
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
Before control
Sgen bus: 3
vm_pu: [1.0050, 1.0114, 1.0072]
sn_mva_limit: [5.000000, 5.000000, 5.000000]
sn_mva: [2.000000, 5.000000, 3.000000]
p_mw: [2.000000, 5.000000, 3.000000]
q_mvar: [0.000000, 0.000000, 0.000000]

After control
Sgen bus: 3
vm_pu: [1.0050, 1.0114, 1.0072]
sn_mva_limit: [5.000000, 5.000000, 5.000000]
sn_mva: [2.000000, 5.000000, 3.000000]
p_mw: [2.000000, 5.000000, 3.000000]
q_mvar: [0.000000, 0.000000, 0.000000]

--- Post control results ---
Max overall vm_pu difference: 7.77305370088918e-09
Max overall va_degree difference: 4.6912902007534285e-08
Max overall i_from_ka difference: 3.736187093039689e-08
Max overall i_to_ka difference: 3.736187093039689e-08
Max overall p_from_mw difference: 3.182370793552991e-07
Max overall p_to_mw difference: 3.2149225681621374e-07
Max overall q_from_mvar difference: 1.3920392468724785e-06
Max overall q_to_mvar difference: 1.403432365029711e-06

bus results
                  vm_pu_mc      vm_pu_d

## Test 6: 4 bus grid with one controlled sgen, asymmetric load and sgen, voltage below tolerance limits at one phase only
Controller to increase voltage towards tolerance band by reactive power infeed, which is the same at all phases (gang control). Possible active power deration. 

In [22]:
net = grids["four_bus_system_asymmetric_voltage_below_tolerance_limits_voltvar_control.pkl"]
mc.run_pf(net)
check_ctr_sgen_stable_state(net, sgen_first_lvl_idx=0, verbose=True)

print("--- Pre/Post control comparison ---")
multiconductor.test.testing_toolbox.print_sgen_stats(net, "", [0], header=False, before_control=True) 
print()
mc.run_pf(net, run_control=True)
multiconductor.test.testing_toolbox.print_sgen_stats(net, "", [0], header=False, before_control=False)
print("\n--- Post control results ---")
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

check_ctr_sgen_stable_state: sgen reactive power (0.0) does not equal target (2.5342866555417034)!
--- Pre/Post control comparison ---
Before control
Sgen bus: 3
vm_pu: [1.0014, 0.9590, 1.0014]
sn_mva_limit: [20.000000, 20.000000, 20.000000]
sn_mva: [5.000000, 1.000000, 5.000000]
p_mw: [5.000000, 1.000000, 5.000000]
q_mvar: [0.000000, 0.000000, 0.000000]

After control
Sgen bus: 3
vm_pu: [1.0089, 0.9670, 1.0089]
sn_mva_limit: [20.000000, 20.000000, 20.000000]
sn_mva: [5.235011, 1.845356, 5.235011]
p_mw: [5.000000, 1.000000, 5.000000]
q_mvar: [1.550915, 1.550915, 1.550915]

--- Post control results ---
Max overall vm_pu difference: 4.488778174716046e-07
Max overall va_degree difference: 4.547266087229218e-06
Max overall i_from_ka difference: 6.285595407717892e-06
Max overall i_to_ka difference: 6.285595407717892e-06
Max overall p_from_mw difference: 2.6994180487349695e-05
Max overall p_to_mw difference: 2.5712457976112546e-05
Max overall q_from_mvar difference: 8.041239567502245e-05
Max

## Test 7: 4 bus grid with one controlled sgen, asymmetric load and sgen, voltage above tolerance limits at one phase only
Controller to decrease voltage towards tolerance band by reactive power infeed, which is the same at all phases (gang control). Possible active power deration. 

In [23]:
net = grids["four_bus_system_asymmetric_voltage_above_tolerance_limits_voltvar_control.pkl"]
mc.run_pf(net)
check_ctr_sgen_stable_state(net, sgen_first_lvl_idx=1, verbose=True)

print("--- Pre/Post control comparison ---")
multiconductor.test.testing_toolbox.print_sgen_stats(net, "", [1], header=False, before_control=True) 
print()
mc.run_pf(net, run_control=True)
multiconductor.test.testing_toolbox.print_sgen_stats(net, "", [1], header=False, before_control=False)
print("\n--- Post control results ---")
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

check_ctr_sgen_stable_state: sgen reactive power (0.0) does not equal target (-7.256197897351095)!
--- Pre/Post control comparison ---
Before control
Sgen bus: 3
vm_pu: [1.0110, 1.0110, 1.0375]
sn_mva_limit: [25.000000, 25.000000, 25.000000]
sn_mva: [5.000000, 5.000000, 25.000000]
p_mw: [5.000000, 5.000000, 25.000000]
q_mvar: [0.000000, 0.000000, 0.000000]

After control
Sgen bus: 3
vm_pu: [0.9935, 0.9935, 1.0196]
sn_mva_limit: [25.000000, 25.000000, 25.000000]
sn_mva: [6.131180, 6.131180, 24.895381]
p_mw: [5.000000, 5.000000, 24.641198]
q_mvar: [-3.548432, -3.548432, -3.548432]

--- Post control results ---
Max overall vm_pu difference: 1.2278751126970633e-07
Max overall va_degree difference: 2.03592682623821e-06
Max overall i_from_ka difference: 6.979181108590637e-07
Max overall i_to_ka difference: 6.979181108590637e-07
Max overall p_from_mw difference: 5.798199097739598e-06
Max overall p_to_mw difference: 5.679937920888278e-06
Max overall q_from_mvar difference: 2.758247573275696e-0

## Test 8: 4 bus grid with one controlled sgen, asymmetric load and sgen, voltage far above tolerance limits
Controller to decrease voltage towards tolerance band by high reactive power infeed, which is the same at all phases (gang control). Possible active power deration.  

In [24]:
net = grids["four_bus_system_asymmetric_voltage_far_above_tolerance_limits_voltvar_control.pkl"]
mc.run_pf(net)
check_ctr_sgen_stable_state(net, sgen_first_lvl_idx=1, verbose=True)

print("--- Pre/Post control comparison ---")
multiconductor.test.testing_toolbox.print_sgen_stats(net, "", [1], header=False, before_control=True) 
print()
mc.run_pf(net, run_control=True)
multiconductor.test.testing_toolbox.print_sgen_stats(net, "", [1], header=False, before_control=False)
print("\n--- Post control results ---")
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

check_ctr_sgen_stable_state: sgen reactive power (0.0) does not equal target (-9.0)!
--- Pre/Post control comparison ---
Before control
Sgen bus: 3
vm_pu: [1.0571, 1.0114, 1.0114]
sn_mva_limit: [30.000000, 30.000000, 30.000000]
sn_mva: [30.000000, 5.000000, 5.000000]
p_mw: [30.000000, 5.000000, 5.000000]
q_mvar: [0.000000, 0.000000, 0.000000]

After control
Sgen bus: 3
vm_pu: [1.0281, 0.9823, 0.9823]
sn_mva_limit: [30.000000, 30.000000, 30.000000]
sn_mva: [29.820249, 7.664629, 7.664629]
p_mw: [29.248944, 5.000000, 5.000000]
q_mvar: [-5.809177, -5.809177, -5.809177]

--- Post control results ---
Max overall vm_pu difference: 5.538196927723504e-07
Max overall va_degree difference: 8.668983566550992e-06
Max overall i_from_ka difference: 3.3377119392774546e-06
Max overall i_to_ka difference: 3.3377119392774546e-06
Max overall p_from_mw difference: 3.363124929478545e-05
Max overall p_to_mw difference: 3.373000951967242e-05
Max overall q_from_mvar difference: 0.00012149717273146621
Max overa

## Test 9: 20 bus grid with four controlled sgen, asymmetric load and sgen, voltage within tolerance limits
This is to test possible interactions between controlled sgens located at different points of the grid. All controllers are expected to converge at valid operation points. 
Expecting no control action. No reactive power infeed from any controlled sgen, no active power change. 

In [25]:
net = grids["twenty_bus_system_voltage_in_tolerance_limits_voltvar_control.pkl"]
mc.run_pf(net)
for i in [0,1,2,3]:
    check_ctr_sgen_stable_state(net, sgen_first_lvl_idx=i, verbose=True)

print("--- Pre/Post control comparison ---")
multiconductor.test.testing_toolbox.print_sgen_stats(net, "", [0,1,2,3], header=False, before_control=True) 
print()
mc.run_pf(net, run_control=True)
multiconductor.test.testing_toolbox.print_sgen_stats(net, "", [0,1,2,3], header=False, before_control=False)
print("\n--- Post control results ---")
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

--- Pre/Post control comparison ---
Before control
Sgen bus: 17 / 10 / 3 / 5
vm_pu: [0.9814, 0.9897, 1.0188] / [1.0255, 1.0320, 1.0243] / [1.0199, 1.0298, 1.0097] / [1.0105, 1.0126, 1.0097]
sn_mva_limit: [0.060000, 0.060000, 0.060000] / [0.060000, 0.060000, 0.060000] / [0.060000, 0.060000, 0.060000] / [0.060000, 0.060000, 0.060000]
sn_mva: [0.032000, 0.035000, 0.031500] / [0.032000, 0.035000, 0.031500] / [0.032000, 0.025000, 0.011500] / [0.032000, 0.035000, 0.031500]
p_mw: [0.032000, 0.035000, 0.031500] / [0.032000, 0.035000, 0.031500] / [0.032000, 0.025000, 0.011500] / [0.032000, 0.035000, 0.031500]
q_mvar: [0.000000, 0.000000, 0.000000] / [0.000000, 0.000000, 0.000000] / [0.000000, 0.000000, 0.000000] / [0.000000, 0.000000, 0.000000]

After control
Sgen bus: 17 / 10 / 3 / 5
vm_pu: [0.9814, 0.9897, 1.0188] / [1.0255, 1.0320, 1.0243] / [1.0199, 1.0298, 1.0097] / [1.0105, 1.0126, 1.0097]
sn_mva_limit: [0.060000, 0.060000, 0.060000] / [0.060000, 0.060000, 0.060000] / [0.060000, 0.060000,

## Test 10: 20 bus grid with four controlled sgen, asymmetric load and sgen, voltage far below tolerance limits
This is to test possible interactions between controlled sgens located at different points of the grid. All controllers are expected to converge at valid operation points. 
Expecting controllers to increase voltage towards tolerance band by high reactive power infeed, which is the same at all phases (gang control). Possible active power deration.  

In [26]:
net = grids["twenty_bus_system_voltage_far_below_tolerance_limits_voltvar_control.pkl"]
mc.run_pf(net)
for i in [0,1,2,3]:
    check_ctr_sgen_stable_state(net, sgen_first_lvl_idx=i, verbose=True)

print("--- Pre/Post control comparison ---")
multiconductor.test.testing_toolbox.print_sgen_stats(net, "", [0,1,2,3], header=False, before_control=True) 
print()
mc.run_pf(net, run_control=True)
multiconductor.test.testing_toolbox.print_sgen_stats(net, "", [0,1,2,3], header=False, before_control=False)
print("\n--- Post control results ---")
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

check_ctr_sgen_stable_state: sgen reactive power (0.0) does not equal target (0.018)!
check_ctr_sgen_stable_state: sgen reactive power (0.0) does not equal target (0.018)!
check_ctr_sgen_stable_state: sgen reactive power (0.0) does not equal target (0.006333746452079356)!
--- Pre/Post control comparison ---
Before control
Sgen bus: 17 / 10 / 3 / 5
vm_pu: [0.8984, 0.8854, 0.9202] / [0.8722, 0.8758, 0.9292] / [0.9597, 0.9515, 0.9603] / [0.9963, 0.9966, 0.9946]
sn_mva_limit: [0.060000, 0.060000, 0.060000] / [0.060000, 0.060000, 0.060000] / [0.060000, 0.060000, 0.060000] / [0.060000, 0.060000, 0.060000]
sn_mva: [0.010000, 0.015000, 0.011500] / [0.005000, 0.010000, 0.011500] / [0.005000, 0.010000, 0.011500] / [0.005000, 0.005000, 0.005000]
p_mw: [0.010000, 0.015000, 0.011500] / [0.005000, 0.010000, 0.011500] / [0.005000, 0.010000, 0.011500] / [0.005000, 0.005000, 0.005000]
q_mvar: [0.000000, 0.000000, 0.000000] / [0.000000, 0.000000, 0.000000] / [0.000000, 0.000000, 0.000000] / [0.000000, 0

## Test 11: 20 bus grid with four controlled sgen, asymmetric load and sgen, voltage below tolerance limits
This is to test possible interactions between controlled sgens located at different points of the grid. All controllers are expected to converge at valid operation points. 
Expecting controllers to increase voltage towards tolerance band by reactive power infeed, which is the same at all phases (gang control). Possible active power deration.  

In [27]:
net = grids["twenty_bus_system_voltage_below_tolerance_limits_voltvar_control.pkl"]
mc.run_pf(net)
for i in [0,1,2,3]:
    check_ctr_sgen_stable_state(net, sgen_first_lvl_idx=i, verbose=True)

print("--- Pre/Post control comparison ---")
multiconductor.test.testing_toolbox.print_sgen_stats(net, "", [0,1,2,3], header=False, before_control=True) 
print()
mc.run_pf(net, run_control=True)
multiconductor.test.testing_toolbox.print_sgen_stats(net, "", [0,1,2,3], header=False, before_control=False)
print("\n--- Post control results ---")
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

check_ctr_sgen_stable_state: sgen reactive power (0.0) does not equal target (0.011873445629737798)!
check_ctr_sgen_stable_state: sgen reactive power (0.0) does not equal target (0.005467579933725387)!
--- Pre/Post control comparison ---
Before control
Sgen bus: 17 / 10 / 3 / 5
vm_pu: [0.9585, 0.9243, 0.9652] / [0.9619, 0.9559, 0.9684] / [0.9897, 0.9944, 1.0106] / [0.9963, 0.9966, 0.9946]
sn_mva_limit: [0.060000, 0.060000, 0.060000] / [0.060000, 0.060000, 0.060000] / [0.060000, 0.060000, 0.060000] / [0.060000, 0.060000, 0.060000]
sn_mva: [0.030000, 0.020000, 0.030000] / [0.005000, 0.010000, 0.011500] / [0.005000, 0.010000, 0.011500] / [0.005000, 0.005000, 0.005000]
p_mw: [0.030000, 0.020000, 0.030000] / [0.005000, 0.010000, 0.011500] / [0.005000, 0.010000, 0.011500] / [0.005000, 0.005000, 0.005000]
q_mvar: [0.000000, 0.000000, 0.000000] / [0.000000, 0.000000, 0.000000] / [0.000000, 0.000000, 0.000000] / [0.000000, 0.000000, 0.000000]

After control
Sgen bus: 17 / 10 / 3 / 5
vm_pu: [0.9

## Test 12: 20 bus grid with four controlled sgen, asymmetric load and sgen, voltage above tolerance limits
This is to test possible interactions between controlled sgens located at different points of the grid. All controllers are expected to converge at valid operation points. 
Expecting controllers to decrease voltage towards tolerance band by reactive power infeed, which is the same at all phases (gang control). Possible active power deration. 

In [28]:
net = grids["twenty_bus_system_voltage_above_tolerance_limits_voltvar_control.pkl"]
mc.run_pf(net)
for i in [0,1,2,3]:
    check_ctr_sgen_stable_state(net, sgen_first_lvl_idx=i, verbose=True)

print("--- Pre/Post control comparison ---")
multiconductor.test.testing_toolbox.print_sgen_stats(net, "", [0,1,2,3], header=False, before_control=True) 
print()
mc.run_pf(net, run_control=True)
multiconductor.test.testing_toolbox.print_sgen_stats(net, "", [0,1,2,3], header=False, before_control=False)
print("\n--- Post control results ---")
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

check_ctr_sgen_stable_state: sgen reactive power (0.0) does not equal target (-0.0012257879739967276)!
check_ctr_sgen_stable_state: sgen reactive power (0.0) does not equal target (-0.0017210964231370652)!
check_ctr_sgen_stable_state: sgen reactive power (0.0) does not equal target (-0.005007593227706826)!
--- Pre/Post control comparison ---
Before control
Sgen bus: 17 / 10 / 3 / 5
vm_pu: [0.9912, 1.0085, 1.0280] / [1.0222, 1.0356, 1.0340] / [1.0290, 1.0255, 1.0414] / [1.0097, 1.0139, 1.0125]
sn_mva_limit: [0.060000, 0.060000, 0.060000] / [0.060000, 0.060000, 0.060000] / [0.060000, 0.060000, 0.060000] / [0.060000, 0.060000, 0.060000]
sn_mva: [0.040000, 0.045000, 0.035000] / [0.030000, 0.040000, 0.035000] / [0.030000, 0.030000, 0.035000] / [0.030000, 0.040000, 0.035000]
p_mw: [0.040000, 0.045000, 0.035000] / [0.030000, 0.040000, 0.035000] / [0.030000, 0.030000, 0.035000] / [0.030000, 0.040000, 0.035000]
q_mvar: [0.000000, 0.000000, 0.000000] / [0.000000, 0.000000, 0.000000] / [0.000000,

## Test 13: 20 bus grid with four controlled sgen, asymmetric load and sgen, voltage far above tolerance limits
This is to test possible interactions between controlled sgens located at different points of the grid. All controllers are expected to converge at valid operation points. 
Expecting controllers to decrease voltage towards tolerance band by high reactive power infeed, which is the same at all phases (gang control). Possible active power deration. 

In [29]:
net = grids["twenty_bus_system_voltage_far_above_tolerance_limits_voltvar_control.pkl"]
mc.run_pf(net)
for i in [0,1,2,3]:
    check_ctr_sgen_stable_state(net, sgen_first_lvl_idx=i, verbose=True)

print("--- Pre/Post control comparison ---")
multiconductor.test.testing_toolbox.print_sgen_stats(net, "", [0,1,2,3], header=False, before_control=True) 
print()
mc.run_pf(net, run_control=True)
multiconductor.test.testing_toolbox.print_sgen_stats(net, "", [0,1,2,3], header=False, before_control=False)
print("\n--- Post control results ---")
c = multiconductor.test.testing_toolbox.compare_to_dss(net)
multiconductor.test.testing_toolbox.print_comparison(c)
print("\nbus results")
print(pd.merge(net.res_bus, net.res_bus_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[["vm_pu_mc", "vm_pu_dss", "va_degree_mc", "va_degree_dss"]])
print("\nline results")
cols = [f"{c}{w}" for c in net.res_line_dss.columns for w in ["_mc", "_dss"]]
print(pd.merge(net.res_line, net.res_line_dss, left_index=True, right_index=True, suffixes=('_mc', '_dss'))[cols])

check_ctr_sgen_stable_state: sgen reactive power (0.0) does not equal target (-0.014357872139811018)!
check_ctr_sgen_stable_state: sgen reactive power (0.0) does not equal target (-0.018)!
check_ctr_sgen_stable_state: sgen reactive power (0.0) does not equal target (-0.018)!
--- Pre/Post control comparison ---
Before control
Sgen bus: 17 / 10 / 3 / 5
vm_pu: [1.0008, 1.0315, 1.0409] / [1.0408, 1.0707, 1.0433] / [1.0415, 1.0651, 1.0546] / [1.0169, 1.0257, 1.0162]
sn_mva_limit: [0.060000, 0.060000, 0.060000] / [0.060000, 0.060000, 0.060000] / [0.060000, 0.060000, 0.060000] / [0.060000, 0.060000, 0.060000]
sn_mva: [0.050000, 0.060000, 0.040000] / [0.050000, 0.060000, 0.040000] / [0.050000, 0.060000, 0.040000] / [0.050000, 0.060000, 0.040000]
p_mw: [0.050000, 0.060000, 0.040000] / [0.050000, 0.060000, 0.040000] / [0.050000, 0.060000, 0.040000] / [0.050000, 0.060000, 0.040000]
q_mvar: [0.000000, 0.000000, 0.000000] / [0.000000, 0.000000, 0.000000] / [0.000000, 0.000000, 0.000000] / [0.000000